In [ ]:
from pathlib import Path

import cfgrib
import numpy as np
import xarray as xr
import xesmf as xe

In [ ]:
ds_ec = xr.open_dataset(Path.home() / "ml-ds_data" / "EC-Earth3.grid.nc")

In [ ]:
coarse_lon = ds_ec["lon"].values
# Shift longitudes to the range [-180, 180] - for era5
coarse_lon = ((coarse_lon + 180) % 360) - 180
coarse_lon = np.sort(coarse_lon)
coarse_lat = ds_ec["lat"].values

In [ ]:
ds_era5_pl = xr.open_dataset(
    sorted((Path.home() / "ml-ds_data" / "ERA5" / "2011" / "pressure-levels").glob("*.grib"))[0]
)

In [ ]:
ds_era5_sl = xr.open_dataset(
    sorted((Path.home() / "ml-ds_data" / "ERA5" / "2011" / "single-levels").glob("*.grib"))[0],
    engine="cfgrib",
    backend_kwargs={"filter_by_keys": {"stepType": "instant"}},
)

In [ ]:
carra_path = sorted((Path.home() / "ml-ds_data" / "CARRA2" / "2011").glob("*.grib"))[0]
ds_carra2 = cfgrib.open_datasets(carra_path)

Land sea mask 'lsm' is somehow has 'orog', which is the same for 'orog',
so they are similar

In [ ]:
ds_carra2_vars = sorted({name for ds in ds_carra2 for name in ds.data_vars})
ds_carra2_vars

This cell only to get a grid, so var name is not important,
but surface roughness 'sr' somehow has another shape, so dont use it

In [ ]:
ds_carra2_var = next(ds for ds in ds_carra2 if "t2m" in ds)


# Regrid ERA5 coarse t2m -> CARRA curvilinear grid
def _pick_coord(ds, candidates):
    for name in candidates:
        if name in ds.coords:
            return ds.coords[name]
        if name in ds:
            return ds[name]
    raise KeyError(f"None of {candidates} found in dataset")


carra_lon = _pick_coord(ds_carra2_var, ["longitude", "lon"])
carra_lat = _pick_coord(ds_carra2_var, ["latitude", "lat"])

# Curvilinear target grid can be provided as 2D lon/lat arrays
grid_out = xr.Dataset({"lon": carra_lon, "lat": carra_lat})
regridder = None

Regird era5 var to carra2 grid, then cut area around Norway

In [ ]:
ds_era5 = ds_era5_sl
ds_era5

In [ ]:
var = "t2m"

# era_time_dim = "valid_time" if "valid_time" in ds_era5[var].dims else "time"
var_era = ds_era5[var]  # .isel({era_time_dim: 0})

In [ ]:
# Interpolate ERA5 -> EC-Earth3 grid (coarse)
var_era_coarse = var_era.interp(
    latitude=coarse_lat,
    longitude=coarse_lon,
    method="linear",
)
var_era_coarse = var_era_coarse.dropna(dim="latitude", how="all").dropna(dim="longitude", how="all")

In [ ]:
# xESMF expects source grid names lon/lat
var_era_coarse = var_era_coarse.rename({"longitude": "lon", "latitude": "lat"})

# Use lazy dask chunks to avoid allocating the full regridded time stack in memory
if "valid_time" in var_era_coarse.dims:
    var_era_coarse = var_era_coarse.chunk({"valid_time": 1})

In [ ]:
if regridder is None:
    regridder = xe.Regridder(
        var_era_coarse, grid_out, method="bilinear", periodic=True, reuse_weights=False
    )
var_era5_on_carra = regridder(
    var_era_coarse, keep_attrs=True, output_chunks={"valid_time": 1}, skipna=True
)

Plotting

In [ ]:
time_step = 100

In [ ]:
var_era5_on_carra.isel(time=time_step).plot(size=10)

In [ ]:
var_carra = ds_carra2_var[var]
var_carra.isel(time=time_step).plot(size=10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(ncols=2, figsize=(8, 11), constrained_layout=True)

var_era5_on_carra.isel(time=time_step, x=slice(2100, 2500), y=slice(400, 1400)).plot(ax=axes[0])
axes[0].set_title("ERA5 t2m on CARRA")

var_carra.isel(time=time_step, x=slice(2100, 2500), y=slice(400, 1400)).plot(ax=axes[1])
axes[1].set_title("CARRA t2m")